In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, subprocess

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = "reasoning anchor"
GOOGLE_DRIVE_PATH = os.path.join("/content/drive", "MyDrive", GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
assert os.path.isdir(GOOGLE_DRIVE_PATH), f"Not found: {GOOGLE_DRIVE_PATH}"

os.chdir(GOOGLE_DRIVE_PATH)
if GOOGLE_DRIVE_PATH not in sys.path:
    sys.path.append(GOOGLE_DRIVE_PATH)

uv_path = shutil.which("uv")
if uv_path is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], check=True)
uv_path = shutil.which("uv")
assert uv_path is not None, "uv install failed"

data_dir = os.path.join(GOOGLE_DRIVE_PATH, "data")
os.makedirs(data_dir, exist_ok=True)

Mounted at /content/drive


In [2]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_dataset = dataset["train"]
test_dataset = dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [3]:
!pip -q install json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.9 MB/s eta 0:00:00


In [4]:
!pip -q install datasets sentence-transformers torch-geometric

import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from torch_geometric.data import Data

from train import train_anchor_model, train
from model import GraphOfThoughtPrunerGraphSAGE

device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
input_dimension = embedding_model.get_sentence_embedding_dimension()

def load_gsm8k_dataset(sample_size=50, split="train"):
    raw_dataset = load_dataset("openai/gsm8k", "main")[split]
    raw_dataset = raw_dataset.select(range(sample_size))
    dataset = []
    for item in raw_dataset:
        dataset.append({"question": item["question"], "answer": item["answer"]})
    return dataset

def build_data_function(graph, anchor_labels=None):
    node_id_to_index = {}
    node_features = []
    for index in range(len(graph["nodes"])):
        node = graph["nodes"][index]
        node_id_to_index[node["node_id"]] = index
        node_features.append(embedding_model.encode(node["text"]))
    edge_source_list = []
    edge_target_list = []
    for edge in graph["edges"]:
        if edge["source"] in node_id_to_index and edge["target"] in node_id_to_index:
            edge_source_list.append(node_id_to_index[edge["source"]])
            edge_target_list.append(node_id_to_index[edge["target"]])
    x = torch.tensor(node_features, dtype=torch.float32)
    edge_index = torch.tensor([edge_source_list, edge_target_list], dtype=torch.long)
    data = Data(x=x, edge_index=edge_index)
    if anchor_labels is not None:
        anchor_label = []
        for node in graph["nodes"]:
            anchor_label.append(anchor_labels.get(node["node_id"], 0.0))
        data.anchor_label = torch.tensor(anchor_label, dtype=torch.float32)
    return data

dataset = load_gsm8k_dataset(sample_size=10)

anchor_model = GraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)

anchor_optimizer = torch.optim.Adam(anchor_model.parameters(), lr=1e-3)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-4)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.0 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1655/2353954843.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  input_dimension = embedding_model.get_sentence_embedding_dimension()


In [6]:
import importlib
import train

importlib.reload(train)

from train import train_anchor_model, train

In [5]:
!pip -q install transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

class HuggingFaceGenerationModel:
    def __init__(self, model_name="Qwen/Qwen3-4B-Instruct-2507", device=None, max_new_tokens=4096):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.device = device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if self.device == "cuda" else torch.float32, device_map="auto" if self.device == "cuda" else None, trust_remote_code=True)
        if self.device != "cuda":
            self.model = self.model.to(self.device)
        self.model.eval()

    def clean_response(self, response):
        response = response.strip()
        if "</think>" in response:
            response = response.split("</think>")[-1].strip()
        if response.startswith("```json"):
            response = response[len("```json"):].strip()
        if response.startswith("```"):
            response = response[len("```"):].strip()
        if response.endswith("```"):
            response = response[:-3].strip()
        return response

    def generate(self, prompt):
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False, pad_token_id=self.tokenizer.eos_token_id)
        generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
        response = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return self.clean_response(response)

generation_model = HuggingFaceGenerationModel("Qwen/Qwen3-4B-Instruct-2507", max_new_tokens=4096)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [7]:
import importlib
import loss
import train
import model

importlib.reload(loss)
importlib.reload(train)
importlib.reload(model)

from train import train_anchor_model, train
from model import GraphOfThoughtPrunerGraphSAGE
from loss import training_loss, anchor_loss, structural_loss, deletion_loss

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class OldGraphOfThoughtPrunerGraphSAGE(nn.Module):
    def __init__(self, input_dimension, hidden_dimension=256, dropout_rate=0.2):
        super().__init__()

        self.input_layer = nn.Linear(input_dimension, hidden_dimension)

        self.first_graphsage_layer = SAGEConv(hidden_dimension, hidden_dimension)
        self.second_graphsage_layer = SAGEConv(hidden_dimension, hidden_dimension)

        self.dropout_layer = nn.Dropout(dropout_rate)

        self.output_layer = nn.Sequential(
            nn.Linear(hidden_dimension, hidden_dimension),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dimension, 1)
        )

    def forward(self, node_features, edge_index):
        hidden = F.relu(self.input_layer(node_features))
        hidden = self.dropout_layer(F.relu(self.first_graphsage_layer(hidden, edge_index)))
        hidden = self.dropout_layer(F.relu(self.second_graphsage_layer(hidden, edge_index)))
        return self.output_layer(hidden).squeeze(-1)

anchor_model = OldGraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)
anchor_model.load_state_dict(torch.load("/content/drive/MyDrive/reasoning anchor/checkpoints/best_anchor_model_2.pt", map_location=device))
anchor_model.eval()

OldGraphOfThoughtPrunerGraphSAGE(
  (input_layer): Linear(in_features=384, out_features=256, bias=True)
  (first_graphsage_layer): SAGEConv(256, 256, aggr=mean)
  (second_graphsage_layer): SAGEConv(256, 256, aggr=mean)
  (dropout_layer): Dropout(p=0.2, inplace=False)
  (output_layer): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=256, out_features=1, bias=True)
  )
)

In [11]:
# train_anchor_model(anchor_model, generation_model, embedding_model, dataset, anchor_optimizer, build_data_function, epochs=30, sample_count=3, device=device, save_path="best_anchor_model_2.pt")

# train(pruning_model, generation_model, embedding_model, dataset, pruning_optimizer, build_data_function, epochs=30, device=device, anchor_model=anchor_model, anchor_threshold=0.5, save_path="best_pruning_model.pt")

/tmp/ipykernel_1315/2353954843.py:37: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  x = torch.tensor(node_features, dtype=torch.float32)


epoch: 0 loss: 0.6940271019935608
epoch: 1 loss: 0.6892229378223419
epoch: 2 loss: 0.6732752382755279
epoch: 3 loss: 0.6156986892223358
epoch: 4 loss: 0.5729258507490158
epoch: 5 loss: 0.5930840313434601
epoch: 6 loss: 0.6737206995487213
epoch: 7 loss: 0.5634238153696061
epoch: 8 loss: 0.5274200916290284
epoch: 9 loss: 0.4689514309167862
epoch: 10 loss: 0.43064717054367063
epoch: 11 loss: 0.3967195272445679
epoch: 12 loss: 0.38216777741909025
epoch: 13 loss: 0.4295033127069473
epoch: 14 loss: 0.4209543287754059
epoch: 15 loss: 0.37938970774412156
epoch: 16 loss: 0.37057117074728013
epoch: 17 loss: 0.3640952467918396
epoch: 18 loss: 0.36540227979421613
epoch: 19 loss: 0.3555284410715103
epoch: 20 loss: 0.3548302918672562
epoch: 21 loss: 0.3675190880894661
epoch: 22 loss: 0.3522632703185081
epoch: 23 loss: 0.3419260308146477
epoch: 24 loss: 0.3432821974158287
epoch: 25 loss: 0.344802188873291
epoch: 26 loss: 0.3419322341680527
epoch: 27 loss: 0.3416531965136528
epoch: 28 loss: 0.34209028

In [ ]:
#train (old version saving) 1e-4,30 -> 1e-5,60 -> 1e-3,60

In [16]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension,block_count = 5).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-4)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.5,0.7,0.9],
    save_path="best_pruning_model_retrial_3_new_gated_1.pt",
    anchor_loss_threshold=0.5
)

anchor_loss: 1.0
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 4.9342932661678034e-05
total_loss: 1.291701911618568
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 0 loss: 1.3443239629268646 evaluation_score: 0.8499882495552437
anchor_loss: 0.33206093311309814
structural_loss: 0.08333333333333333
deletion_loss: 0.8571428571428571
deletion_weight: 0.2397696762880493
total_loss: 0.620911131836188
kept_nodes: 6 / 7
kept_edges: 6 / 7
epoch: 1 loss: 1.0646239519119263 evaluation_score: 0.7967884380234038
anchor_loss: 1.0
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 4.9342932661678034e-05
total_loss: 1.291701911618568
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 2 loss: 1.197169977426529 evaluation_score: 0.8442635648136408
anchor_loss: 1.0
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 4.9342932661678034e-05
total_loss: 1.291701911618568
kept_nodes: 5 / 7
kept_edges: 5 / 7
epo

In [17]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension,block_count = 4).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-3)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.5,0.7,0.9],
    save_path="best_pruning_model_retrial_3_new_gated_2.pt",
    anchor_loss_threshold=0.5
)

anchor_loss: 0.6472973823547363
structural_loss: 0.5
deletion_loss: 0.5
deletion_weight: 0.0002089986335573014
total_loss: 1.147401881671515
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 0 loss: 1.4447527408599854 evaluation_score: 0.9621540862054114
anchor_loss: 0.33206093311309814
structural_loss: 0.5416666666666667
deletion_loss: 0.6428571428571428
deletion_weight: 0.003213179445468028
total_loss: 0.8757932151375658
kept_nodes: 5 / 7
kept_edges: 4 / 7
epoch: 1 loss: 1.1538186073303223 evaluation_score: 0.828288172265954
anchor_loss: 0.33206093311309814
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 0.03778680780429236
total_loss: 0.6507181767828308
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 2 loss: 0.9025180459022522 evaluation_score: 0.8263154450182596
anchor_loss: 0.33206093311309814
structural_loss: 0.3333333333333333
deletion_loss: 0.5714285714285714
deletion_weight: 0.025235537034558404
total_loss: 0.679814573323322
kept_nodes: 4 / 7
kept_ed

In [18]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension,block_count = 4).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-4)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.5,0.7,0.9],
    save_path="best_pruning_model_retrial_3_new_gated_3.pt",
    anchor_loss_threshold=0.5
)

anchor_loss: 0.33206093311309814
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 0.03778680780429236
total_loss: 0.6507181767828308
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 0 loss: 0.880460548400879 evaluation_score: 0.845028700109327
anchor_loss: 0.33206093311309814
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 0.03778680780429236
total_loss: 0.6507181767828308
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 1 loss: 1.3607473820447922 evaluation_score: 0.8613902675015145
anchor_loss: 0.33206093311309814
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 0.03778680780429236
total_loss: 0.6507181767828308
kept_nodes: 5 / 7
kept_edges: 5 / 7
epoch: 2 loss: 1.2241820096969604 evaluation_score: 0.8572192608073957
anchor_loss: 0.33206093311309814
structural_loss: 0.2916666666666667
deletion_loss: 0.7142857142857143
deletion_weight: 0.03778680780429236
total_loss: 0.65071817678283

In [19]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension,block_count = 4).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-5)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.5,0.7,0.9],
    save_path="best_pruning_model_retrial_3_new_gated_4.pt",
    anchor_loss_threshold=0.5
)

anchor_loss: 0.33206093311309814
structural_loss: 0.5416666666666666
deletion_loss: 0.42857142857142855
deletion_weight: 0.003213179445468028
total_loss: 0.8751046766849654
kept_nodes: 4 / 7
kept_edges: 2 / 7
epoch: 0 loss: 1.447946047782898 evaluation_score: 0.8011923779095611
anchor_loss: 0.33206093311309814
structural_loss: 0.5416666666666666
deletion_loss: 0.42857142857142855
deletion_weight: 0.003213179445468028
total_loss: 0.8751046766849654
kept_nodes: 4 / 7
kept_edges: 2 / 7
epoch: 1 loss: 1.0008140921592712 evaluation_score: 0.7807952655996506
anchor_loss: 0.33206093311309814
structural_loss: 0.5416666666666666
deletion_loss: 0.42857142857142855
deletion_weight: 0.003213179445468028
total_loss: 0.8751046766849654
kept_nodes: 4 / 7
kept_edges: 2 / 7
epoch: 2 loss: 0.8460846722126008 evaluation_score: 0.7807952655996506
anchor_loss: 0.33206093311309814
structural_loss: 0.5416666666666666
deletion_loss: 0.42857142857142855
deletion_weight: 0.003213179445468028
total_loss: 0.87510

In [20]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension,block_count = 2).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-4)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.5,0.7,0.9],
    save_path="best_pruning_model_retrial_3_new_gated_5.pt",
    anchor_loss_threshold=0.5
)

anchor_loss: 0.33206093311309814
structural_loss: 0.41666666666666663
deletion_loss: 0.5
deletion_weight: 0.011126068352992296
total_loss: 0.754290633956261
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 0 loss: 0.9768840372562408 evaluation_score: 0.8405547739583685
anchor_loss: 0.33206093311309814
structural_loss: 0.41666666666666663
deletion_loss: 0.5
deletion_weight: 0.011126068352992296
total_loss: 0.754290633956261
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 1 loss: 0.9477217078208924 evaluation_score: 0.834595940370775
anchor_loss: 0.33206093311309814
structural_loss: 0.41666666666666663
deletion_loss: 0.5
deletion_weight: 0.011126068352992296
total_loss: 0.754290633956261
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 2 loss: 0.9413345694541931 evaluation_score: 0.8308250807011882
anchor_loss: 0.33206093311309814
structural_loss: 0.41666666666666663
deletion_loss: 0.5
deletion_weight: 0.011126068352992296
total_loss: 0.754290633956261
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 3 loss: 1

In [15]:
import json
import os

os.chdir("/content/drive/MyDrive/reasoning anchor")

got_anchor_path = "/content/drive/MyDrive/reasoning anchor/data/gsm8k_got_anchor_qwen_long.jsonl"

got_anchor_dataset = []
with open(got_anchor_path, "r", encoding="utf-8") as file:
    for line in file:
        got_anchor_dataset.append(json.loads(line))

print("loaded got_anchor_dataset:", len(got_anchor_dataset))

import os
import torch
import random
import numpy as np
from loss import training_loss, anchor_loss, structural_loss, deletion_loss
from train import prune_graph_by_top_ratio, get_anchors_from_anchor_model
from model import GraphOfThoughtPrunerGraphSAGE

random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)

eval_dataset = got_anchor_dataset[:20]
keep_ratios = [0.5, 0.7, 0.9]
anchor_threshold = 0.5

checkpoint_specs = {
    "gnn_3_layer_1e3": {
        "block_count": 3,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt"
        ]
    },
    "gnn_2_layer_1e3": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt"
        ]
    },
    "gnn_5_layer_1e4": {
        "block_count": 5,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_3_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt"
        ]
    },
    "gnn_4_layer_1e3": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_4_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt"
        ]
    },
    "gnn_4_layer_1e4": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_5_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt"
        ]
    },
    "gnn_4_layer_1e5": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_6_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt"
        ]
    },
    "gnn_2_layer_1e4": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt"
        ]
    }
}

gnn_models = {}

for model_name, spec in checkpoint_specs.items():
    checkpoint_path = None
    for path in spec["paths"]:
        if os.path.exists(path):
            checkpoint_path = path
            break

    if checkpoint_path is None:
        print("skip missing checkpoint:", model_name)
        continue

    loaded_model = GraphOfThoughtPrunerGraphSAGE(
        input_dimension,
        block_count=spec["block_count"]
    ).to(device)

    loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    loaded_model.eval()
    gnn_models[model_name] = loaded_model
    print("loaded:", model_name, checkpoint_path)

results = {}
methods = ["no_pruning", "random_pruning"] + list(gnn_models.keys())

for method in methods:
    method_score = 0.0
    method_anchor_loss = 0.0
    method_structural_loss = 0.0
    method_deletion_loss = 0.0
    method_node_ratio = 0.0
    method_edge_ratio = 0.0
    method_count = 0

    for line in eval_dataset:
        question = line["question"]
        graph = line["graph"]

        anchors = []
        if "anchor_labels" in line:
            for node in graph["nodes"]:
                if line["anchor_labels"].get(node["node_id"], 0.0) >= anchor_threshold:
                    anchors.append(node)
        else:
            data = build_data_function(graph)
            if device is not None:
                data = data.to(device)
            anchors = get_anchors_from_anchor_model(anchor_model, graph, data, anchor_threshold)

        for keep_ratio in keep_ratios:
            if method == "no_pruning":
                pruned_graph = graph

            elif method == "random_pruning":
                keep_count = max(1, int(len(graph["nodes"]) * keep_ratio))
                keep_indices = random.sample(range(len(graph["nodes"])), keep_count)
                keep_node_ids = []
                for index in keep_indices:
                    keep_node_ids.append(graph["nodes"][index]["node_id"])

                pruned_graph = {
                    "nodes": [],
                    "edges": []
                }

                for node in graph["nodes"]:
                    if node["node_id"] in keep_node_ids:
                        pruned_graph["nodes"].append(node)

                for edge in graph["edges"]:
                    if edge["source"] in keep_node_ids and edge["target"] in keep_node_ids:
                        pruned_graph["edges"].append(edge)

            else:
                data = build_data_function(graph)
                if device is not None:
                    data = data.to(device)

                current_model = gnn_models[method]
                current_model.eval()

                with torch.no_grad():
                    node_keep_score = current_model(data.x, data.edge_index)
                    node_keep_probability = torch.sigmoid(node_keep_score)

                pruned_graph = prune_graph_by_top_ratio(graph, node_keep_probability, keep_ratio)

            a_loss = anchor_loss(pruned_graph, graph, question, generation_model, embedding_model)
            s_loss = structural_loss(pruned_graph, graph, anchors)
            d_loss = deletion_loss(pruned_graph, graph)
            score = training_loss(pruned_graph, graph, question, generation_model, embedding_model, anchors)

            node_ratio = len(pruned_graph["nodes"]) / len(graph["nodes"]) if len(graph["nodes"]) > 0 else 0.0
            edge_ratio = len(pruned_graph["edges"]) / len(graph["edges"]) if len(graph["edges"]) > 0 else 0.0

            method_score += score
            method_anchor_loss += a_loss
            method_structural_loss += s_loss
            method_deletion_loss += d_loss
            method_node_ratio += node_ratio
            method_edge_ratio += edge_ratio
            method_count += 1

    results[method] = {
        "evaluation_score": method_score / method_count,
        "anchor_loss": method_anchor_loss / method_count,
        "structural_loss": method_structural_loss / method_count,
        "deletion_loss": method_deletion_loss / method_count,
        "kept_node_ratio": method_node_ratio / method_count,
        "kept_edge_ratio": method_edge_ratio / method_count
    }

results

loaded got_anchor_dataset: 100
loaded: gnn_3_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt
loaded: gnn_2_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt
loaded: gnn_5_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt
loaded: gnn_4_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt
loaded: gnn_4_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt
loaded: gnn_4_layer_1e5 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


loaded: gnn_2_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt


/tmp/ipykernel_1655/2353954843.py:37: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  x = torch.tensor(node_features, dtype=torch.float32)


{'no_pruning': {'evaluation_score': np.float64(0.8403722919769422),
  'anchor_loss': 0.40783472107723356,
  'structural_loss': 0.09290327380952382,
  'deletion_loss': 1.0,
  'kept_node_ratio': 1.0,
  'kept_edge_ratio': 1.0},
 'random_pruning': {'evaluation_score': np.float64(1.0207320683536854),
  'anchor_loss': 0.47018121868992846,
  'structural_loss': 0.513860104016354,
  'deletion_loss': 0.564805395734637,
  'kept_node_ratio': 0.6676293995859213,
  'kept_edge_ratio': 0.46198139188335274},
 'gnn_3_layer_1e3': {'evaluation_score': np.float64(0.8413912466978342),
  'anchor_loss': 0.4056487189916273,
  'structural_loss': 0.21800112734487734,
  'deletion_loss': 0.9025984375456882,
  'kept_node_ratio': 0.9138306922546052,
  'kept_edge_ratio': 0.8913661828367707},
 'gnn_2_layer_1e3': {'evaluation_score': np.float64(0.8919514707214028),
  'anchor_loss': 0.4408500747444729,
  'structural_loss': 0.2904220478595479,
  'deletion_loss': 0.7605764944951328,
  'kept_node_ratio': 0.8234627992780168

In [21]:
import json
import os
import torch
import random
import numpy as np

os.chdir("/content/drive/MyDrive/reasoning anchor")

got_anchor_path = "/content/drive/MyDrive/reasoning anchor/data/gsm8k_got_anchor_qwen_long.jsonl"

got_anchor_dataset = []
with open(got_anchor_path, "r", encoding="utf-8") as file:
    for line in file:
        got_anchor_dataset.append(json.loads(line))

print("loaded got_anchor_dataset:", len(got_anchor_dataset))

from loss import training_loss, anchor_loss, structural_loss, deletion_loss
from train import prune_graph_by_top_ratio, get_anchors_from_anchor_model
from model import GraphOfThoughtPrunerGraphSAGE

random.seed(1234)
np.random.seed(1234)
torch.manual_seed(1234)

eval_dataset = got_anchor_dataset[:20]
keep_ratios = [0.5, 0.7, 0.9]
anchor_threshold = 0.5

checkpoint_specs = {
    "gnn_3_layer_1e3": {
        "block_count": 3,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt"
        ]
    },
    "gnn_2_layer_1e3": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt"
        ]
    },
    "gnn_5_layer_1e4": {
        "block_count": 5,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_3_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt"
        ]
    },
    "gnn_4_layer_1e3": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_4_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt"
        ]
    },
    "gnn_4_layer_1e4": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_5_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt"
        ]
    },
    "gnn_4_layer_1e5": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_6_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt"
        ]
    },
    "gnn_2_layer_1e4": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/checkpoints/best_pruning_model_retrial_2_new_2.pt",
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt"
        ]
    },

    "gnn_5_layer_1e4_new_gated_1": {
        "block_count": 5,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_1.pt"
        ]
    },
    "gnn_4_layer_1e3_new_gated_2": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_2.pt"
        ]
    },
    "gnn_4_layer_1e4_new_gated_3": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_3.pt"
        ]
    },
    "gnn_4_layer_1e5_new_gated_4": {
        "block_count": 4,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_4.pt"
        ]
    },
    "gnn_2_layer_1e4_new_gated_5": {
        "block_count": 2,
        "paths": [
            "/content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_5.pt"
        ]
    }
}

gnn_models = {}

for model_name, spec in checkpoint_specs.items():
    checkpoint_path = None

    for path in spec["paths"]:
        if os.path.exists(path):
            checkpoint_path = path
            break

    if checkpoint_path is None:
        print("skip missing checkpoint:", model_name)
        continue

    loaded_model = GraphOfThoughtPrunerGraphSAGE(
        input_dimension,
        block_count=spec["block_count"]
    ).to(device)

    loaded_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    loaded_model.eval()

    gnn_models[model_name] = loaded_model
    print("loaded:", model_name, checkpoint_path)

results = {}
methods = ["no_pruning", "random_pruning"] + list(gnn_models.keys())

for method in methods:
    method_score = 0.0
    method_anchor_loss = 0.0
    method_structural_loss = 0.0
    method_deletion_loss = 0.0
    method_node_ratio = 0.0
    method_edge_ratio = 0.0
    method_count = 0

    for line in eval_dataset:
        question = line["question"]
        graph = line["graph"]

        anchors = []
        if "anchor_labels" in line:
            for node in graph["nodes"]:
                if line["anchor_labels"].get(node["node_id"], 0.0) >= anchor_threshold:
                    anchors.append(node)
        else:
            data = build_data_function(graph)
            if device is not None:
                data = data.to(device)

            anchors = get_anchors_from_anchor_model(
                anchor_model,
                graph,
                data,
                anchor_threshold
            )

        for keep_ratio in keep_ratios:
            if method == "no_pruning":
                pruned_graph = graph

            elif method == "random_pruning":
                keep_count = max(1, int(len(graph["nodes"]) * keep_ratio))
                keep_indices = random.sample(range(len(graph["nodes"])), keep_count)

                keep_node_ids = []
                for index in keep_indices:
                    keep_node_ids.append(graph["nodes"][index]["node_id"])

                pruned_graph = {
                    "nodes": [],
                    "edges": []
                }

                for node in graph["nodes"]:
                    if node["node_id"] in keep_node_ids:
                        pruned_graph["nodes"].append(node)

                for edge in graph["edges"]:
                    if edge["source"] in keep_node_ids and edge["target"] in keep_node_ids:
                        pruned_graph["edges"].append(edge)

            else:
                data = build_data_function(graph)
                if device is not None:
                    data = data.to(device)

                current_model = gnn_models[method]
                current_model.eval()

                with torch.no_grad():
                    node_keep_score = current_model(data.x, data.edge_index)
                    node_keep_probability = torch.sigmoid(node_keep_score)

                pruned_graph = prune_graph_by_top_ratio(
                    graph,
                    node_keep_probability,
                    keep_ratio
                )

            a_loss = anchor_loss(
                pruned_graph,
                graph,
                question,
                generation_model,
                embedding_model
            )

            s_loss = structural_loss(
                pruned_graph,
                graph,
                anchors
            )

            d_loss = deletion_loss(
                pruned_graph,
                graph
            )

            score = training_loss(
                pruned_graph,
                graph,
                question,
                generation_model,
                embedding_model,
                anchors
            )

            node_ratio = len(pruned_graph["nodes"]) / len(graph["nodes"]) if len(graph["nodes"]) > 0 else 0.0
            edge_ratio = len(pruned_graph["edges"]) / len(graph["edges"]) if len(graph["edges"]) > 0 else 0.0

            method_score += score
            method_anchor_loss += a_loss
            method_structural_loss += s_loss
            method_deletion_loss += d_loss
            method_node_ratio += node_ratio
            method_edge_ratio += edge_ratio
            method_count += 1

    results[method] = {
        "evaluation_score": method_score / method_count,
        "anchor_loss": method_anchor_loss / method_count,
        "structural_loss": method_structural_loss / method_count,
        "deletion_loss": method_deletion_loss / method_count,
        "kept_node_ratio": method_node_ratio / method_count,
        "kept_edge_ratio": method_edge_ratio / method_count
    }

results

loaded got_anchor_dataset: 100
loaded: gnn_3_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial.pt
loaded: gnn_2_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2.pt
loaded: gnn_5_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_2.pt
loaded: gnn_4_layer_1e3 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_4_new_2.pt
loaded: gnn_4_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_5_new_2.pt
loaded: gnn_4_layer_1e5 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_6_new_2.pt
loaded: gnn_2_layer_1e4 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_2_new_2.pt
loaded: gnn_5_layer_1e4_new_gated_1 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_1.pt
loaded: gnn_4_layer_1e3_new_gated_2 /content/drive/MyDrive/reasoning anchor/best_pruning_model_retrial_3_new_gated_2.pt
loaded: gnn_4_layer_1e4_new_g

{'no_pruning': {'evaluation_score': np.float64(0.8403722919769422),
  'anchor_loss': 0.40783472107723356,
  'structural_loss': 0.09290327380952382,
  'deletion_loss': 1.0,
  'kept_node_ratio': 1.0,
  'kept_edge_ratio': 1.0},
 'random_pruning': {'evaluation_score': np.float64(1.0207320683536854),
  'anchor_loss': 0.47018121868992846,
  'structural_loss': 0.513860104016354,
  'deletion_loss': 0.564805395734637,
  'kept_node_ratio': 0.6676293995859213,
  'kept_edge_ratio': 0.46198139188335274},
 'gnn_3_layer_1e3': {'evaluation_score': np.float64(0.8413912466978342),
  'anchor_loss': 0.4056487189916273,
  'structural_loss': 0.21800112734487734,
  'deletion_loss': 0.9025984375456882,
  'kept_node_ratio': 0.9138306922546052,
  'kept_edge_ratio': 0.8913661828367707},
 'gnn_2_layer_1e3': {'evaluation_score': np.float64(0.8919514707214028),
  'anchor_loss': 0.4408500747444729,
  'structural_loss': 0.2904220478595479,
  'deletion_loss': 0.7605764944951328,
  'kept_node_ratio': 0.8234627992780168

In [29]:
pruning_model = GraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)
pruning_optimizer = torch.optim.Adam(pruning_model.parameters(), lr=1e-3)

train(
    pruning_model,
    generation_model,
    embedding_model,
    dataset,
    pruning_optimizer,
    build_data_function,
    epochs=30,
    device=device,
    anchor_model=anchor_model,
    anchor_threshold=0.3,
    keep_ratios=[0.4,0.6,0.8],
    save_path="best_pruning_model_retrial_2.pt",
    anchor_loss_threshold=0.4
)

anchor_loss: 0.33206093311309814
structural_loss: 0.5833333333333334
deletion_loss: 0.21428571428571427
deletion_weight: 0.0021205803104977342
total_loss: 0.9158486765129668
kept_nodes: 2 / 7
kept_edges: 1 / 7
epoch: 0 loss: 0.9295761585235596 evaluation_score: 1.0649720119757584
anchor_loss: 0.6472973823547363
structural_loss: 0.5416666666666666
deletion_loss: 0.3571428571428571
deletion_weight: 0.0001377902040735372
total_loss: 1.1890132598085719
kept_nodes: 3 / 7
kept_edges: 2 / 7
epoch: 1 loss: 1.1823418974876403 evaluation_score: 1.0314397639138209
anchor_loss: 0.33206093311309814
structural_loss: 0.7083333333333333
deletion_loss: 0.5
deletion_weight: 0.000608477071424013
total_loss: 1.0406985049821433
kept_nodes: 4 / 7
kept_edges: 3 / 7
epoch: 2 loss: 1.093818074464798 evaluation_score: 0.804844876989104
anchor_loss: 0.33206093311309814
structural_loss: 0.75
deletion_loss: 0.42857142857142855
deletion_weight: 0.00040121599786362605
total_loss: 1.0822328828264682
kept_nodes: 4 / 7

In [22]:
from train import prune_graph_by_top_ratio, parse_response, get_anchors_from_anchor_model, print_loss_parts
from prompt import build_prompt_reasoning_trace, build_prompt_build_graph
from loss import anchor_loss

line = dataset[0]
question = line["question"]

if "graph" in line:
    graph = line["graph"]
else:
    reasoning_trace = generation_model.generate(build_prompt_reasoning_trace(question))
    graph = parse_response(generation_model.generate(build_prompt_build_graph(question, reasoning_trace)))
    line["graph"] = graph

data = build_data_function(graph)
if device is not None:
    data = data.to(device)

pruning_model.eval()
with torch.no_grad():
    node_keep_score = pruning_model(data.x, data.edge_index)
    node_keep_probability = torch.sigmoid(node_keep_score)

pruned_graph = prune_graph_by_top_ratio(graph, node_keep_probability, keep_ratio=0.7)

current_anchor_loss = anchor_loss(pruned_graph, graph, question, generation_model, embedding_model)

if current_anchor_loss > 0.3 and "anchor_labels" in line:
    anchors = []
    for node in graph["nodes"]:
        if line["anchor_labels"].get(node["node_id"], 0.0) >= 0.7:
            anchors.append(node)
else:
    anchors = get_anchors_from_anchor_model(anchor_model, graph, data, 0.7)

print("PRUNED NODES")
print(pruned_graph["nodes"])

print("PRUNED EDGES")
print(pruned_graph["edges"])

print("ORIGINAL EDGES")
print(graph["edges"])

print("ANCHORS")
print(anchors)

print_loss_parts(pruned_graph, graph, question, generation_model, embedding_model, anchors)

PRUNED NODES
[{'node_id': 'N1', 'step_index': 1, 'text': 'Natalia sold 48 clips in April.', 'function_tags': ['problem_setup'], 'depends_on': [], 'endpoint': False, 'endpoint_status': None, 'endpoint_type': None}, {'node_id': 'N3', 'step_index': 3, 'text': 'Half of 48 is calculated as 48 ÷ 2 = 24.', 'function_tags': ['active_computation'], 'depends_on': ['N1'], 'endpoint': False, 'endpoint_status': None, 'endpoint_type': None}, {'node_id': 'N4', 'step_index': 4, 'text': 'Therefore, Natalia sold 24 clips in May.', 'function_tags': ['result_consolidation'], 'depends_on': ['N3'], 'endpoint': False, 'endpoint_status': None, 'endpoint_type': None}, {'node_id': 'N5', 'step_index': 5, 'text': 'The total number of clips sold in April and May is 48 + 24.', 'function_tags': ['result_consolidation'], 'depends_on': ['N1', 'N4'], 'endpoint': False, 'endpoint_status': None, 'endpoint_type': None}, {'node_id': 'N6', 'step_index': 6, 'text': '48 + 24 = 72.', 'function_tags': ['active_computation'], 'd

In [23]:
import os
import json
import torch
import numpy as np
from datasets import load_dataset
from torch_geometric.data import Data
from model import GraphOfThoughtPrunerGraphSAGE
from prompt import build_prompt_reasoning_trace, build_prompt_build_graph
from train import parse_response, find_anchors_monte_carlo

os.chdir("/content/drive/MyDrive/reasoning anchor")
device = "cuda" if torch.cuda.is_available() else "cpu"

data_dir = "/content/drive/MyDrive/reasoning anchor/data"
checkpoint_dir = "/content/drive/MyDrive/reasoning anchor/checkpoints"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

sample_size = 100
sample_count = 5
save_path = "/content/drive/MyDrive/reasoning anchor/data/gsm8k_got_anchor_qwen_long.jsonl"
result_path = "/content/drive/MyDrive/reasoning anchor/data/anchor_model_eval_results.json"

def build_data_function(graph, anchor_labels=None):
    node_id_to_index = {}
    node_features = []
    for index in range(len(graph["nodes"])):
        node = graph["nodes"][index]
        node_id_to_index[node["node_id"]] = index
        node_features.append(embedding_model.encode(node["text"]))
    edge_source_list = []
    edge_target_list = []
    for edge in graph["edges"]:
        if edge["source"] in node_id_to_index and edge["target"] in node_id_to_index:
            edge_source_list.append(node_id_to_index[edge["source"]])
            edge_target_list.append(node_id_to_index[edge["target"]])
    x = torch.tensor(np.array(node_features), dtype=torch.float32)
    edge_index = torch.tensor([edge_source_list, edge_target_list], dtype=torch.long)
    data = Data(x=x, edge_index=edge_index)
    if anchor_labels is not None:
        anchor_label = []
        for node in graph["nodes"]:
            anchor_label.append(anchor_labels.get(node["node_id"], 0.0))
        data.anchor_label = torch.tensor(anchor_label, dtype=torch.float32)
    return data

raw_dataset = load_dataset("openai/gsm8k", "main")["train"]
raw_dataset = raw_dataset.select(range(sample_size))
raw_dataset = [{"question": item["question"], "answer": item["answer"]} for item in raw_dataset]

if os.path.exists(save_path):
    got_anchor_dataset = []
    with open(save_path, "r", encoding="utf-8") as file:
        for line in file:
            got_anchor_dataset.append(json.loads(line))
else:
    got_anchor_dataset = []

start_index = len(got_anchor_dataset)
print("resume from:", start_index, "/", len(raw_dataset))

for index in range(start_index, len(raw_dataset)):
    line = raw_dataset[index]
    question = line["question"]
    try:
        reasoning_trace = generation_model.generate(build_prompt_reasoning_trace(question))
        graph = parse_response(generation_model.generate(build_prompt_build_graph(question, reasoning_trace)))
        anchor_labels = find_anchors_monte_carlo(generation_model, graph, question, embedding_model, sample_count=sample_count)
        item = {
            "question": question,
            "answer": line["answer"],
            "reasoning_trace": reasoning_trace,
            "graph": graph,
            "anchor_labels": anchor_labels
        }
        got_anchor_dataset.append(item)
        with open(save_path, "w", encoding="utf-8") as file:
            for saved_item in got_anchor_dataset:
                file.write(json.dumps(saved_item, ensure_ascii=False) + "\n")
        print("saved:", index + 1, "/", len(raw_dataset))
    except Exception as error:
        print("failed index:", index, "error:", error)

print("total saved:", len(got_anchor_dataset))

checkpoint_candidates = [
    "/content/drive/MyDrive/reasoning anchor/checkpoints/best_anchor_model_2.pt",
    "/content/drive/MyDrive/reasoning anchor/best_anchor_model_2.pt"
]

checkpoint_path = None
for path in checkpoint_candidates:
    if os.path.exists(path):
        checkpoint_path = path
        break

if checkpoint_path is None:
    raise FileNotFoundError("Cannot find best_anchor_model_2.pt in checkpoints or project root.")

anchor_model = GraphOfThoughtPrunerGraphSAGE(input_dimension).to(device)
anchor_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
anchor_model.eval()
print("loaded checkpoint:", checkpoint_path)

def evaluate_anchor_model_accuracy(anchor_model, dataset, build_data_function, device=None, threshold=0.5):
    anchor_model.eval()
    correct = 0
    total = 0
    true_positive = 0
    false_positive = 0
    false_negative = 0
    with torch.no_grad():
        for line in dataset:
            if "graph" not in line or "anchor_labels" not in line:
                continue
            graph = line["graph"]
            anchor_labels = line["anchor_labels"]
            data = build_data_function(graph, anchor_labels)
            if device is not None:
                data = data.to(device)
            anchor_score = anchor_model(data.x, data.edge_index)
            anchor_probability = torch.sigmoid(anchor_score)
            anchor_prediction = anchor_probability >= threshold
            anchor_target = data.anchor_label >= threshold
            for index in range(len(anchor_target)):
                prediction = bool(anchor_prediction[index].item())
                target = bool(anchor_target[index].item())
                if prediction == target:
                    correct += 1
                if prediction is True and target is True:
                    true_positive += 1
                if prediction is True and target is False:
                    false_positive += 1
                if prediction is False and target is True:
                    false_negative += 1
                total += 1
    accuracy = correct / total if total > 0 else 0.0
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive > 0 else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return {
        "threshold": threshold,
        "total_nodes": total,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

results = {}
for threshold in [0.3, 0.5, 0.7]:
    result = evaluate_anchor_model_accuracy(anchor_model, got_anchor_dataset, build_data_function, device=device, threshold=threshold)
    results[str(threshold)] = result
    print(result)

with open(result_path, "w", encoding="utf-8") as file:
    json.dump(results, file, ensure_ascii=False, indent=2)

print("saved eval results:", result_path)

resume from: 0 / 100
saved: 1 / 100
saved: 2 / 100
saved: 3 / 100
saved: 4 / 100
saved: 5 / 100
saved: 6 / 100
saved: 7 / 100
saved: 8 / 100
saved: 9 / 100
saved: 10 / 100
saved: 11 / 100
saved: 12 / 100
saved: 13 / 100
saved: 14 / 100
saved: 15 / 100
saved: 16 / 100
saved: 17 / 100
saved: 18 / 100
saved: 19 / 100
saved: 20 / 100
saved: 21 / 100
saved: 22 / 100
saved: 23 / 100
saved: 24 / 100
saved: 25 / 100
saved: 26 / 100
saved: 27 / 100
saved: 28 / 100
saved: 29 / 100
saved: 30 / 100
saved: 31 / 100
saved: 32 / 100
saved: 33 / 100
saved: 34 / 100
saved: 35 / 100
saved: 36 / 100
saved: 37 / 100
saved: 38 / 100
saved: 39 / 100
saved: 40 / 100
saved: 41 / 100
saved: 42 / 100
saved: 43 / 100
saved: 44 / 100
saved: 45 / 100
saved: 46 / 100
saved: 47 / 100
saved: 48 / 100
saved: 49 / 100
saved: 50 / 100
saved: 51 / 100
saved: 52 / 100
saved: 53 / 100
saved: 54 / 100
saved: 55 / 100
saved: 56 / 100
saved: 57 / 100
saved: 58 / 100
saved: 59 / 100
saved: 60 / 100
saved: 61 / 100
saved: 62 / 